# 38. Flask Micro-framework — `flask`
**Python Libraries Lab · Phase 3 · Web & Networking**

_Small, simple web apps and APIs, one route at a time._

⚙️ **Runs in Colab** after the setup cell below.

Flask is a micro framework. It gives you the core of a web app, routes, requests, responses and templates, and lets you add everything else only when you need it. A complete Flask app can fit in ten lines, which makes it a favourite for small sites, internal tools and APIs, and for learning how the web works.

**In this notebook:** You will build a ShopEasy mini store. A home page from a template, a JSON API to list, fetch and create products, a form that adds a product, and proper error codes. You will test it with Flask's test client, then run it live in Colab.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q flask
import importlib.metadata as md
for pkg in ['flask']:
    print(pkg, md.version(pkg))

## Step 1 · A whole app in a few lines
Each `@app.route` connects a URL to a function. Data lives in a list for now.

In [ ]:
from flask import Flask, jsonify, request, render_template_string, redirect, url_for, abort

app = Flask(__name__)
products = [{"id": 1, "name": "Steel Bottle", "price": 499}, {"id": 2, "name": "Yoga Mat", "price": 1299}]

PAGE = """<h1>ShopEasy</h1>
<ul>{% for p in products %}<li>{{ p.name }}: Rs. {{ p.price }}</li>{% endfor %}</ul>
<form method="post" action="/add"><input name="name" placeholder="Name"> <input name="price" type="number" placeholder="Price"> <button>Add</button></form>"""

@app.route("/")
def home():
    return render_template_string(PAGE, products=products)

@app.post("/add")
def add_from_form():
    products.append({"id": len(products) + 1, "name": request.form["name"], "price": int(request.form["price"])})
    return redirect(url_for("home"))
print("Routes:", [r.rule for r in app.url_map.iter_rules() if r.endpoint != "static"])

## Step 2 · A JSON API
List, get one (with 404), and create (with 201 and validation).

In [ ]:
@app.get("/api/products")
def list_products():
    max_price = request.args.get("max_price", type=int)
    items = [p for p in products if max_price is None or p["price"] <= max_price]
    return jsonify(items)

@app.get("/api/products/<int:pid>")
def get_product(pid):
    for p in products:
        if p["id"] == pid:
            return jsonify(p)
    abort(404, description="Product not found")

@app.post("/api/products")
def create_product():
    data = request.get_json(silent=True) or {}
    if not data.get("name") or not isinstance(data.get("price"), (int, float)) or data["price"] <= 0:
        return jsonify(error="name and a positive price are required"), 422
    item = {"id": max(p["id"] for p in products) + 1, "name": data["name"], "price": data["price"]}
    products.append(item)
    return jsonify(item), 201

@app.errorhandler(404)
def not_found(e):
    return jsonify(error=str(e.description)), 404

## Step 3 · Test it without a browser
`app.test_client()` sends fake requests straight to your app.

In [ ]:
client = app.test_client()
print(client.get("/api/products").json)
print(client.get("/api/products/2").json)
r = client.get("/api/products/99"); print(r.status_code, r.json)
r = client.post("/api/products", json={"name": "Desk Lamp", "price": 899}); print(r.status_code, r.json)
r = client.post("/api/products", json={"name": "Free Lamp", "price": -5}); print(r.status_code, r.json)
r = client.post("/add", data={"name": "Notebook", "price": "149"}); print("form:", r.status_code, "->", r.headers["Location"])
print(client.get("/").get_data(as_text=True)[:160])

## Step 4 · Run the real server
A background thread keeps the notebook free. Then open it in Colab.

In [ ]:
import threading
from werkzeug.serving import make_server
flask_server = make_server("0.0.0.0", 5000, app)
threading.Thread(target=flask_server.serve_forever, daemon=True).start()

import requests
print(requests.get("http://127.0.0.1:5000/api/products").json())
def open_in_browser(port, path="/"):
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(port, path=path)
    except ImportError:
        print(f"Not in Colab: open http://localhost:{port}{path} in your browser")
open_in_browser(5000)

## Step 5 · Stop the server
Shut it down cleanly.

In [ ]:
flask_server.shutdown()
print("Flask server stopped")

## ✍️ Your turn
Use `client` to **POST** a new product `{"name": "Gel Pens", "price": 99}` to `/api/products`, and store the **id** of the created product in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    ok = answer is not None and products[-1]["name"] == "Gel Pens" and answer == products[-1]["id"]
    return ok, f"Expected the new product's id, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = client.post("/api/products", json={"name": "Gel Pens", "price": 99}).json["id"]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Route**
- **Request / response**
- **Template**
- **JSON API**
- **Status code**
- **Test client**

**Next:** 39. FastAPI (`fastapi`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/39_fastapi.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install flask
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.